# H₂ simulation: two vibrational modes

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
from openfermion.ops import FermionOperator
from openfermion.transforms import bravyi_kitaev
from qutip import Qobj, tensor, basis, sesolve, expect, qeye, destroy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ===== 1. 手动构造 H₂ 分子哈密顿量（sto-3g 基组, R=0.74Å） =====
fermionic_ham = FermionOperator()
fermionic_ham += FermionOperator('', -0.81261)
fermionic_ham += FermionOperator('0^ 0', -1.25248)
fermionic_ham += FermionOperator('1^ 1', -0.47595)
fermionic_ham += FermionOperator('2^ 2', -0.47595)
fermionic_ham += FermionOperator('3^ 3', -1.25248)
fermionic_ham += FermionOperator('0^ 1^ 1 0', 0.67449)
fermionic_ham += FermionOperator('0^ 2^ 2 0', 0.66458)
fermionic_ham += FermionOperator('0^ 3^ 3 0', 0.18129)
fermionic_ham += FermionOperator('1^ 2^ 2 1', 0.18129)
fermionic_ham += FermionOperator('1^ 3^ 3 1', 0.66458)
fermionic_ham += FermionOperator('2^ 3^ 3 2', 0.67449)
fermionic_ham += FermionOperator('0^ 1^ 2 3', 0.66347)
fermionic_ham += FermionOperator('3^ 2^ 1 0', 0.66347)


In [ ]:
# ===== 2. BK 映射为泡利哈密顿量 =====
qubit_ham = bravyi_kitaev(fermionic_ham)
n_qubits = 4

def qubit_operator_to_qutip(qubit_op, n_qubits):
    pauli_map = {'X': Qobj([[0, 1], [1, 0]]),
                 'Y': Qobj([[0, -1j], [1j, 0]]),
                 'Z': Qobj([[1, 0], [0, -1]]),
                 'I': Qobj([[1, 0], [0, 1]])}
    H = 0
    for term, coeff in qubit_op.terms.items():
        ops = [pauli_map['I']] * n_qubits
        for idx, op in term:
            ops[idx] = pauli_map[op]
        H += coeff * tensor(ops)
    return H

H_el = qubit_operator_to_qutip(qubit_ham, n_qubits)


In [ ]:
# ===== 3. 构建双振动模式（简谐振子） =====
n1, n2 = 10, 8
w1, w2 = 1.0, 0.7
a1, a2 = destroy(n1), destroy(n2)
H_vib = tensor(w1 * a1.dag() * a1, qeye(n2)) + tensor(qeye(n1), w2 * a2.dag() * a2)
H_vib_ext = tensor(qeye(2 ** n_qubits), H_vib)
H_vib_ext.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]

In [ ]:
# ===== 4. 构建耦合项：Z0/Z1 ⊗ (a + a†) =====
g1, g2 = 0.5, 0.3
Z = Qobj([[1, 0], [0, -1]])
Z0 = tensor(Z, *[qeye(2) for _ in range(n_qubits - 1)])
Z1 = tensor(qeye(2), Z, *[qeye(2) for _ in range(n_qubits - 2)])
Z0 = tensor(Z0, qeye(n1), qeye(n2))
Z1 = tensor(Z1, qeye(n1), qeye(n2))
Z0.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]
Z1.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]

a_term1 = tensor(qeye(2**n_qubits), a1 + a1.dag(), qeye(n2))
a_term2 = tensor(qeye(2**n_qubits), qeye(n1), a2 + a2.dag())
a_term1.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]
a_term2.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]

H_int = g1 * Z0 * a_term1 + g2 * Z1 * a_term2


In [ ]:
# ===== 5. 构造总哈密顿量 H_tot =====
H_el_ext = tensor(H_el, qeye(n1), qeye(n2))
H_el_ext.dims = [[2]*n_qubits + [n1, n2], [2]*n_qubits + [n1, n2]]
H_tot = H_el_ext + H_vib_ext + H_int

In [ ]:
# ===== 6. 初始态 =====
psi_el = tensor([basis(2, 0) for _ in range(n_qubits)])
psi_vib = tensor(basis(n1, 0), basis(n2, 0))
psi0 = tensor(psi_el, psi_vib)

In [ ]:
# ===== 7. 动力学演化与观测 =====
tlist = np.linspace(0, 25, 300)
result = sesolve(H_tot, psi0, tlist)

Z0_obs = tensor(Z0, qeye(n1), qeye(n2))
z_expect = expect(Z0_obs, result.states)

plt.plot(tlist, z_expect)
plt.xlabel("Time")
plt.ylabel("⟨Z₀⟩")
plt.title("H₂ with Two Vibronic Modes")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ===== 输出电子本征能级 =====
e_vals = H_el.eigenenergies()
print("Lowest 4 electronic eigenenergies (manual H₂):")
print(np.sort(e_vals)[:4])